# Run the HydroCNHS FE Walter zero-shot example

## Goal
Repository YAML → hydrocnhs.Model.run → Dam observation → preserved prompt/schema → requested release → shared physics → downstream routing → audit/trace. The adapter supplies decision logic; HydroCNHS runs the simulation using frozen inflow. Offline replies are synthetic, not model outputs. The actual prompt is prompts/fe_walter_reference/zeroshot_system.txt.

## Setup
Python 3.10+ with repository dependencies. All cells use local files; no provider calls.

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: clean-upgrade-99ea43e42a


## Steps

### Inspect input, system prompt and response fixture

In [2]:
print((ROOT/'examples/fe_walter/zero_shot/experiment.tutorial.yaml').read_text())
print((ROOT/'prompts/fe_walter_reference/zeroshot_system.txt').read_text())
fixture=json.loads((ROOT/'tests/fixtures/zeroshot_responses.json').read_text())
print('Reply kind:', fixture['data_kind']); print('First reply:', fixture['responses'][0])

# FE Walter V3.7 zeroshot: original prompt builders, synthetic replies, zero model calls.
schema_version: llm-reservoir-experiment.v1
case: ../../../cases/fe_walter_reference/case.json # Frozen hydrology sample; V3.7 physics.
run_id: fe-walter-tutorial-zeroshot        # Change with output before a new run.
output: ../../../runs/fe-walter-tutorial-zeroshot
max_calls: 0                             # This example cannot call a provider.
auditors:
  release_limits:
    type: release_bounds
    version: "1"
agents:
  fe_operator:
    reservoir: FEWalter                  # Reservoir ID in case.json.
    mode: offline                        # Native prompt/audit run on fixture replies.
    strategy: reference_v3_7             # Generic strategy selector; optional vendor plugin.
    reference:
      profile: fe_walter                 # This reference profile is FE-specific.
      method: zeroshot                   # Original zero-shot or DKP policy.
      system_prompt: ../../../prompts/fe_wal

### Run daily and inspect rendered trace

In [3]:
settings, verified=ensure_offline('examples/fe_walter/zero_shot/experiment.tutorial.yaml')
print(json.dumps(verified,indent=2))
raw=[json.loads(p.read_text()) for p in (Path(settings.report['output'])/'raw').glob('policy_*.json')]
first=min(raw,key=lambda item:item['backend_trace']['input_variables']['timestep'])
print('Daily variables:',first['backend_trace']['input_variables'])
print('Schema fields:', list(first['backend_trace']['schema']['properties']))


{
  "status": "PASS",
  "days": 3,
  "reservoir_count": 1,
  "reservoir_days": 3,
  "date_start": "2016-01-01",
  "date_end": "2016-01-03",
  "audit_counts": {
    "PASS": 3
  },
  "max_reservoir_balance_residual_m3": 0.0,
  "max_system_balance_residual_m3": 0.0,
  "max_routing_difference_cms": 0.0,
  "final_channel_storage_m3": 0.0,
  "observed_network_calls": 0,
  "operating_strategy_accepted": null,
  "audit_binding_counts": {
    "versioned_suite": 3,
    "legacy_unbound": 0
  },
  "external_audit_pins_checked": []
}
Daily variables: {'date': '2016-01-01', 'history_semantics': 'prior requested and implemented outflows; appended after physical commit', 'inflow_cfs': 1134.4949071013637, 'inflow_cms': 32.12531870666853, 'native_history': [], 'pool_elevation_start_ft': 1300.0, 'reservoir_profile': 'fe_walter', 'schema_version': 'reservoir-daily-inputs.v1', 'storage_start_af': 1992.61, 'storage_start_m3': 2457848.279729185, 'timestep': 0}
Schema fields: ['applicable_target_ft', 'intend

## Checks
Review the printed inputs, counts, source labels and residuals. Synthetic replies validate interfaces only.

## Next steps
A new reservoir may use an explicit reservoir profile; it must supply its own facts and level-storage relation. Never copy FE Walter capacities or target levels to it.